In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]

#### Abaixo, fazemos query sobre a tabela chamados_hora para termos uma visão geral sobre cada coluna

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Abertura_Chamado is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select count(*) from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Finalizacao_Atendimento is null

#### Apenas na coluna Hora_Inicio_Atendimento foram encontrados valores nulos. Vamos verificar o motivo desses valores nulos estarem aparecendo. A primeira hipótese é de que esses chamados foram abertos, mas o cliente desistiu de dar prosseguimento ao atendimento, e por isso esse valor de início está nulo. Se esse for o caso, o horário de finalização deve ser igual ao horário de abertura. Além disso, a coluna resolvido da tabela chamados deve ser falsa.  
####  
#### A segunda hipótese é de que esses chamados não são por atendimento humano, e o horário de início, apesar de não estar declarado, deve ser igual ao horário de abertura.

In [0]:
%sql
-- essa query verifica se há casos resolvidos com horário de início nulo
select * from vcredit_medallion.bronze.chamados_hora a 
inner join vcredit_medallion.silver.tb_chamados b
on a.ID_Chamado=b.id_chamado
where a.Hora_Inicio_Atendimento is null and b.resolvido="true"


In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora a 
inner join vcredit_medallion.silver.tb_chamados b
on a.ID_Chamado=b.id_chamado
where a.Hora_Abertura_Chamado=a.Hora_Finalizacao_Atendimento

#### As queries acima nos permitem concluir que a primeira hipótese está correta. Os casos de horário de início de atendimento nulo são aqueles em que o chamado não foi resolvido, uma vez que não existe caso de horário de início nulo e chamado resolvido.  
#### Além disso, a segunda query refuta a segunda hipótese, de que o horário de abertura deveria ser igual ao de finalização para os casos de horário de início nulo.

#### As tabelas que necessitam formatação são: Hora_Abertura_Chamado, Hora_Inicio_Atendimento e Hora_Finalizacao_Atendimento

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento like "%-%"

In [0]:
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.chamados_hora")

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
# dd/mm/aaaa  <qualquer coisa que não seja dígito>  hh:mm
# o \D+ cobre " às ", " �s ", " " etc. — não depende do encoding do "à"
PADRAO_DATAHORA = r"(\d{1,2})/(\d{1,2})/(\d{4})\D+(\d{1,2}):(\d{2})"

def extrair_datahora(col, prefixo):
    """
    De strings tipo '12/07/2025 às 09:23:50' extrai:
      {prefixo}_data   -> date (yyyy-MM-dd); data impossível (ex.: 31/02) vira null
      {prefixo}_hora   -> int 0-23; fora da faixa vira null
      {prefixo}_minuto -> int 0-59; fora da faixa vira null
    Retorna um dict para usar com df.withColumns(). Se a string não bater com o padrão, tudo vem null.
    """
    c = F.col(col) if isinstance(col, str) else col
    s = c.cast("string")
    dia, mes, ano, hora, minuto = [F.regexp_extract(s, PADRAO_DATAHORA, i) for i in range(1, 6)]

    # sem match, regexp_extract devolve "" -> texto "--" -> try_to_timestamp devolve null
    data = F.to_date(F.try_to_timestamp(F.concat_ws("-", ano, mes, dia), F.lit("yyyy-M-d")))
    h = normalizar_inteiro(hora)
    m = normalizar_inteiro(minuto)

    return {
        f"{prefixo}_data":   data,
        f"{prefixo}_hora":   F.when(h.between(0, 23), h).cast("int"),
        f"{prefixo}_minuto": F.when(m.between(0, 59), m).cast("int"),
    }

In [0]:

# ---------- limites válidos para as datas ----------
DATA_MINIMA = date(2025, 1, 1)
DATA_MAXIMA = date.today()                     # dia em que o notebook roda

# ordem de preferência dos vizinhos para cada evento: o evento mais próximo primeiro
VIZINHOS = {
    "abertura": ["inicio", "fim"],
    "inicio":   ["abertura", "fim"],
    "fim":      ["inicio", "abertura"],
}

def data_no_intervalo(prefixo):
    """A data do evento, se estiver dentro dos limites; senão null."""
    d = F.col(f"{prefixo}_data")
    return F.when(d.between(F.lit(DATA_MINIMA), F.lit(DATA_MAXIMA)), d)

def esta_fora(prefixo):
    """True quando a data existe mas está fora dos limites (null não conta como fora)."""
    d = F.col(f"{prefixo}_data")
    return d.isNotNull() & ~d.between(F.lit(DATA_MINIMA), F.lit(DATA_MAXIMA))

# ==================================
# 1. Extrai data, hora e minuto das três colunas de horário.
#    Data impossível, hora fora de 0-23 e minuto fora de 0-59 já viram null em extrair_datahora.
# ==================================
df_extraido = (
    df_bronze_chamados_hora
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumns(extrair_datahora("Hora_Abertura_Chamado", "abertura"))
    .withColumns(extrair_datahora("Hora_Inicio_Atendimento", "inicio"))
    .withColumns(extrair_datahora("Hora_Finalizacao_Atendimento", "fim"))
    .drop("Hora_Abertura_Chamado", "Hora_Inicio_Atendimento", "Hora_Finalizacao_Atendimento", "ingestion_datetime")
)

# data substituta de cada evento: a primeira data válida entre os vizinhos
substituta = {p: F.coalesce(*[data_no_intervalo(v) for v in viz]) for p, viz in VIZINHOS.items()}

# ==================================
# 2. Diagnóstico: quantas datas estão fora do intervalo e quantas dá para corrigir
# ==================================
contagem = df_extraido.agg(*[
    expr
    for p in VIZINHOS
    for expr in (
        F.sum(F.when(esta_fora(p), 1).otherwise(0)).alias(f"{p}_fora"),
        F.sum(F.when(esta_fora(p) & substituta[p].isNotNull(), 1).otherwise(0)).alias(f"{p}_corrigida"),
    )
]).first()

print(f"Intervalo válido: {DATA_MINIMA} até {DATA_MAXIMA}")
for p in VIZINHOS:
    fora = contagem[f"{p}_fora"] or 0
    corrigida = contagem[f"{p}_corrigida"] or 0
    print(f"  {p}: {fora} fora do intervalo -> {corrigida} corrigidas por outra data da linha, "
          f"{fora - corrigida} viraram null")

# ==================================
# 3. Correção: data fora do intervalo é trocada pela substituta (ou null, se nenhum vizinho for válido).
# ==================================
df_silver_chamados_hora = df_extraido.withColumns({
    f"{p}_data": F.when(esta_fora(p), substituta[p]).otherwise(F.col(f"{p}_data"))
    for p in VIZINHOS
})

# virada de meia-noite: mesmo dia do evento anterior, mas horário menor -> foi no dia seguinte
def minutos_do_dia(p):
    return F.col(f"{p}_hora") * 60 + F.col(f"{p}_minuto")

for anterior, posterior in [("abertura", "inicio"), ("inicio", "fim")]:
    virou_o_dia = (
        (F.col(f"{posterior}_data") == F.col(f"{anterior}_data"))
        & (minutos_do_dia(posterior) < minutos_do_dia(anterior))
    )
    df_silver_chamados_hora = df_silver_chamados_hora.withColumn(
        f"{posterior}_data",
        F.when(virou_o_dia, F.date_add(f"{posterior}_data", 1)).otherwise(F.col(f"{posterior}_data")),
    )
    
display(df_silver_chamados_hora)

In [0]:
# ======================= grava na silver =======================
(df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora"))
print(f"\nGravado em {silver_schema}.tb_chamados_hora")